# 🧪 Laboratório 4 — Regressão inversa: prescrever um sítio... e auditar a prescrição

**Minicurso LaPSEE · UNESP Ilha Solteira**

Nas Partes II–V a regressão **descrevia**: $y$ era um preço e perguntávamos o que o move.
Na Parte VI ela passa a **prescrever**: $y = s$ é um índice de aptidão para geração
distribuída (GD), e a mesma matriz de efeitos $S = \beta\,(\mathbb I-\rho W)^{-1}$ pontua
uma intervenção que **nós escolhemos** — uma GD na barra $j$. Isso é a *regressão inversa*.

Mas este laboratório tem uma segunda metade que nenhum slide tem: a **prova dos nove**. O
mercado do IEEE 39 (o mesmo do Laboratório 2) nos permite *construir de verdade* os 50 MW de
GD em cada uma das 39 barras, rodar o OPF e medir quanto o sistema economiza. 

A estatística pode ser una ferramenta que pode ser usada para prescreves, mas é sempre importante ter presente que trabalhamos sob um sistema físico e nossas decisões devem ser auditadas

## Regras do jogo

1. **Duplas**: prever antes de rodar. 2. `# ╔══ COMPLETAR` + **VERIFICAÇÃO** como sempre.
3. As 📝 alimentam o **entregável**: a recomendação de sítio com dupla evidência.

| Bloco | Tema | Tempo |
|---|---|---|
| A | o sistema brinquedo  (caminho 1–2–3–4) | 10 min |
| B | o índice de aptidão e a regressão inversa no IEEE 39 | 25 min |
| C | a prova dos nove: 39 usinas hipotéticas no OPF | 20 min |
| D | dois objetivos, uma recomendação | 15 min |

---
## 0 · Setup (Apenas rodar)

Reaproveitamos o **mercado do IEEE 39 do Laboratório 2** (bolsão importador caro a leste da
fronteira de Fiedler, resto barato, limites a 70%) — agora com um parâmetro extra: `dg`
permite **construir uma usina** de capacidade e custo dados em qualquer barra.

In [ ]:
import json
import logging
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import scipy.stats as st
import pandapower as pp
import pandapower.networks as pn
from libpysal.weights import W as PysalW
import spreg

logging.getLogger("pandapower").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
AZUL, VERMELHO, CINZA, VERDE = "#1F3A5F", "#C8102E", "#a0a0a0", "#2a9d2a"

BOLSAO = [14, 15, 16, 18, 19, 20, 21, 22, 23, 26, 32, 33, 34, 35]
CUSTO_CARO, CUSTO_BARATO = 45.0, 18.0


def mercado_case39(esc_limites=0.7, dg=None):
    "O mercado do Lab 2. dg = (barra, MW, custo) constrói uma usina nova."
    net = pn.case39()
    net.line["max_i_ka"] = net.line["max_i_ka"] * esc_limites
    net.line["max_loading_percent"] = 100.0
    net.trafo["max_loading_percent"] = 100.0
    net.poly_cost.drop(net.poly_cost.index, inplace=True)
    barras_g = [int(b) for b in net.gen.bus] + [int(net.ext_grid.bus.iloc[0])]
    ofertas = {g: (CUSTO_CARO if g in BOLSAO else CUSTO_BARATO) for g in barras_g}
    pp.create_poly_cost(net, 0, "ext_grid",
                        cp1_eur_per_mw=ofertas[int(net.ext_grid.bus.iloc[0])])
    for gi, g in net.gen.iterrows():
        pp.create_poly_cost(net, gi, "gen", cp1_eur_per_mw=ofertas[int(g.bus)])
    if dg is not None:
        b, p_mw, custo = dg
        gi = pp.create_gen(net, bus=int(b), p_mw=0.0, min_p_mw=0.0,
                           max_p_mw=float(p_mw), controllable=True)
        pp.create_poly_cost(net, gi, "gen", cp1_eur_per_mw=float(custo))
    return net


def coords_das_barras(net):
    return {int(b): tuple(json.loads(g)["coordinates"][:2])
            for b, g in net.bus["geo"].items()}


def grafo_topologico(net):
    G = nx.Graph(); G.add_nodes_from(sorted(net.bus.index))
    for _, ln in net.line.iterrows():
        G.add_edge(int(ln.from_bus), int(ln.to_bus))
    for _, tr in net.trafo.iterrows():
        G.add_edge(int(tr.hv_bus), int(tr.lv_bus))
    return G


def mapa_da_rede(net, valores, titulo="", cmap="plasma", marcas=()):
    "Rede pintada por um vetor de valores por barra; `marcas` = barras destacadas."
    pos = coords_das_barras(net); G = grafo_topologico(net)
    barras = sorted(net.bus.index)
    fig, ax = plt.subplots(figsize=(10.5, 6.2))
    for u, v in G.edges():
        (x1, y1), (x2, y2) = pos[u], pos[v]
        ax.plot([x1, x2], [y1, y2], color="lightgray", lw=1.0, zorder=1)
    sc = ax.scatter([pos[b][0] for b in barras], [pos[b][1] for b in barras],
                    c=pd.Series(valores).reindex(barras).values, cmap=cmap,
                    s=210, edgecolors="k", zorder=3)
    for b in marcas:
        ax.scatter(*pos[b], marker="s", s=440, facecolors="none",
                   edgecolors=VERDE, linewidths=2.6, zorder=4)
    for b in barras:
        ax.annotate(str(b), pos[b], ha="center", va="center", fontsize=6.5,
                    color="w", zorder=5)
    plt.colorbar(sc)
    ax.set_title(titulo); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.tight_layout(); plt.show()


net = mercado_case39()
pp.rundcopp(net)
C0 = float(net.res_cost)
barras = sorted(net.bus.index)
print(f"Mercado pronto — custo base C0 = {C0:,.0f} $/h")

---
## Bloco A · O Sistema brinquedo

O caminho $1\!-\!2\!-\!3\!-\!4$ com $W$ padronizada por linhas e $\rho = 0.5$. Construam
$A_{spatial} = (\mathbb I - \rho W)^{-1}$ e extraiam as duas pontuações da aula:

- **diagonal** $(A)_{jj}$ → valor *próprio* (miópico) de melhorar a barra $j$;
- **soma da coluna** $\sum_i (A)_{ij}$ → valor *social* (o que $j$ **emite** para o sistema).

In [ ]:
W_toy = np.array([[0, 1, 0, 0],
                  [1, 0, 1, 0],
                  [0, 1, 0, 1],
                  [0, 0, 1, 0]], dtype=float)
W_toy = W_toy / W_toy.sum(axis=1, keepdims=True)        # padronização por linhas

# ╔══ COMPLETAR ══ dica: A_toy = np.linalg.inv(np.eye(4) - 0.5*W_toy);
# ║   diagonal com .diagonal(); somas de coluna com .sum(axis=0); de linha, axis=1
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática (os números EXATOS do slide) ──────────────────────
assert np.allclose(proprio_toy, [1.156, 1.244, 1.244, 1.156], atol=0.005), \
    "a diagonal deveria ser (1.156, 1.244, 1.244, 1.156)"
assert np.allclose(social_toy, [1.6, 2.4, 2.4, 1.6], atol=0.005), \
    "as somas de coluna deveriam ser (1.6, 2.4, 2.4, 1.6)"
assert np.allclose(linhas_toy, 2.0), "somas de linha: todas 1/(1-ρ) = 2"
tab = pd.DataFrame({"barra": [1, 2, 3, 4], "próprio (diag)": proprio_toy.round(3),
                    "social (col)": social_toy.round(3), "linha": linhas_toy.round(3)})
print("✅ VERIFICAÇÃO OK — os números do slide, reproduzidos:")
display(tab)

💡 **Resultado contraintuitivo nº1 .** As somas de
**linha** são todas $1/(1-\rho) = 2$: toda barra *recebe* a mesma influência total. As de
**coluna** não: as barras do meio (2 e 3) *emitem* 50% mais que as pontas. O valor social de
um sítio é propriedade da **posição na rede** — e a pontuação míope (diagonal, que varia só
8%) é quase cega a isso.

📝 **Pergunta A:** por que a diagonal varia tão pouco (1.156 vs 1.244) enquanto a coluna
varia tanto (1.6 vs 2.4)? O que cada uma conta fisicamente?

> *Resposta de vocês aqui:* ______



---
## Bloco B · O índice de aptidão e a regressão inversa no IEEE 39

### B.0 — As matérias-primas (entregue): painel de 48 h e quatro regressores

Os ingredientes do índice, todos já conhecidos: o **painel de LMPs** (48 h com perfil
diário), a **exposição à congestão** (frequência de saturação das linhas incidentes), a
**densidade de demanda** (carga a ≤ 3 saltos), o **potencial renovável** (proxy das
coordenadas) e a **betweenness** (Lab 1).

In [ ]:
def perfil(n=48, pico=0.97, vale=0.70, seed=2026):
    g = np.random.default_rng(seed); t = np.arange(n)
    return 0.5 * (pico + vale) + 0.5 * (pico - vale) * np.sin(2 * np.pi * (t - 6) / 24) \
           + g.normal(scale=0.015, size=n)


cargas_base = net.load["p_mw"].values.copy()
lmps, loadings = [], []
for m in perfil():
    net.load["p_mw"] = cargas_base * m
    try:
        pp.rundcopp(net)
        lmps.append(net.res_bus["lam_p"].reindex(barras).values)
        loadings.append(net.res_line["loading_percent"].values)
    except Exception:                                   # hora infactível -> NaN
        lmps.append(np.full(len(barras), np.nan))
        loadings.append(np.full(len(net.line), np.nan))
net.load["p_mw"] = cargas_base
painel = pd.DataFrame(lmps, columns=barras).dropna()
loadings = np.array(loadings)
print(f"Painel: {painel.shape[0]} horas válidas | "
      f"ramos saturados em média: {(np.nan_to_num(loadings) > 99.9).sum(axis=1).mean():.1f}")


In [ ]:

# --- os quatro regressores + o preço médio --------------------------------------
G39 = grafo_topologico(net)
btw = pd.Series(nx.betweenness_centrality(G39, normalized=True)).reindex(barras)
carga_b = net.load.groupby("bus")["p_mw"].sum().reindex(barras, fill_value=0.0)
densidade = pd.Series(
    [float(carga_b.reindex(list(nx.single_source_shortest_path_length(G39, b, cutoff=3)),
                           fill_value=0.0).sum()) for b in barras], index=barras)
frec_sat = (np.nan_to_num(loadings) > 99.9).mean(axis=0)
exposicao = pd.Series(0.0, index=barras)
for li, f in enumerate(frec_sat):
    if f > 0:
        exposicao[int(net.line.loc[li, "from_bus"])] += f
        exposicao[int(net.line.loc[li, "to_bus"])] += f
cps = coords_das_barras(net)
xs = np.array([cps[b][0] for b in barras]); ys = np.array([cps[b][1] for b in barras])
renovavel = pd.Series(0.5 + 0.5 * (1 - (xs - xs.min()) / np.ptp(xs))
                      * (1 - (ys - ys.min()) / np.ptp(ys)), index=barras)
lmp_medio = painel.mean(axis=0).reindex(barras)

ingredientes = pd.DataFrame({"lmp_medio": lmp_medio, "betweenness": btw,
                             "densidade": densidade, "exposicao": exposicao,
                             "renovavel": renovavel})
ingredientes.describe().round(2)

### B.1 — O índice de aptidão $s$

Como na aula (Faria et al. 2016): $s$ = soma ponderada dos componentes **padronizados**
(z-scores), com pesos $0.35\,$(preço)$\; / \;0.20\,$(densidade)$\; / \;0.20\,$(exposição)
$\; / \;0.25\,$(renovável). A betweenness fica de fora do índice — ela será regressor no SAR.

In [ ]:
# ╔══ COMPLETAR ══ dica: z = (col - col.mean())/col.std(ddof=0) para cada componente;
# ║   s = 0.35*z_lmp + 0.20*z_dens + 0.20*z_expo + 0.25*z_renov  (uma pd.Series)
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert abs(s.mean()) < 1e-9 and 0.3 < s.std() < 1.2, "z-scores ponderados: média 0"
assert set(s.nlargest(3).index) <= {2, 13, 14, 15, 16, 17, 25, 26}, \
    "o pódio do índice deveria estar no corredor congestionado ou na fronteira do bolsão"
print(f"✅ VERIFICAÇÃO OK — top-5 do índice: {s.nlargest(5).index.tolist()}")
mapa_da_rede(net, s, "O índice de aptidão s — brilham o corredor congestionado (barra 2) "
                     "e a fronteira do bolsão", cmap="plasma")

### B.2 — A regressão inversa: do SAR aos dois rankings

Ajustem o SAR do índice sobre os quatro regressores da aula
($b$, $D$, $E$, $r$), recuperem $\hat\rho$, construam
$\hat A = (\mathbb I - \hat\rho W)^{-1}$ (com $W$ = adjacência padronizada) e extraiam as
duas pontuações do Bloco A — agora para 39 barras de verdade.

In [ ]:
W39 = np.zeros((len(barras), len(barras)))
for u, v in G39.edges():
    W39[u, v] = W39[v, u] = 1.0


def a_pysal(Wm, ids):
    soma = Wm.sum(axis=1); soma = np.where(soma == 0, 1.0, soma)
    Wn = Wm / soma[:, None]
    return PysalW({ids[i]: [ids[j] for j in np.where(Wn[i] > 0)[0]] for i in range(len(ids))},
                  {ids[i]: [Wn[i, j] for j in np.where(Wn[i] > 0)[0]] for i in range(len(ids))},
                  silence_warnings=True)


NOMES_X = ["betweenness", "densidade", "exposicao", "renovavel"]
X = ingredientes[NOMES_X].values

# ╔══ COMPLETAR ══ dica: sar = spreg.ML_Lag(s.values.reshape(-1,1), X, w=a_pysal(W39, barras),
# ║   method="full", name_x=NOMES_X); rho_hat = float(sar.rho);
# ║   A_hat = np.linalg.inv(np.eye(39) - rho_hat * W_padronizada) com
# ║   W_padronizada = W39/W39.sum(axis=1, keepdims=True);
# ║   próprio = diagonal de A_hat; social = somas de COLUNA de A_hat
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert 0.40 < rho_hat < 0.80 and float(sar.z_stat[-1][0]) > 3, \
    "o índice deveria exibir spillover forte e significativo"
assert np.allclose(A_hat.sum(axis=1), 1 / (1 - rho_hat), atol=1e-8), \
    "somas de linha = 1/(1-ρ), como no brinquedo"
assert social.idxmax() != proprio.idxmax(), "os dois rankings deveriam divergir"
print(f"✅ VERIFICAÇÃO OK — ρ̂ = {rho_hat:.3f} (z = {float(sar.z_stat[-1][0]):.1f})")
print(f"top-5 SOCIAL (emissores): {social.nlargest(5).index.tolist()}")
print(f"top-5 PRÓPRIO (míope)   : {proprio.nlargest(5).index.tolist()}")
graus = pd.Series({b: G39.degree(b) for b in barras})
print(f"grau médio do top-5 social: {graus[social.nlargest(5).index].mean():.1f} "
      f"vs grau médio da rede: {graus.mean():.1f}")

💡 **Resultado contraintuitivo nº2.** compare os resultados dos dois top 5: 



📝 **Pergunta B:** o ranking social diz que a barra 15 é a melhor emissora. Antes do Bloco C,
façam a aposta: ao construir 50 MW de GD barata, a barra 15 será também a que **mais reduz o
custo do sistema** no OPF? Explique.

> *Resposta de vocês aqui:* ______



---
## Bloco C · A prova dos nove: 39 usinas hipotéticas

A estatística prescreveu. Agora a física audita: para **cada** barra $j$, construímos de
verdade uma GD de **50 MW a \$5/MWh** (solar com custo quase nulo), rodamos o OPF e medimos
$\Delta C_j = C_0 - C(\text{GD em } j)$ — o valor *verdadeiro* do sítio em \$/h.

In [ ]:
dC = {}
for b in barras:
    n2 = mercado_case39(dg=(b, 50.0, 5.0))
    try:
        pp.rundcopp(n2)
        dC[b] = C0 - float(n2.res_cost)
    except Exception:
        dC[b] = np.nan
dC = pd.Series(dC, name="dC")

print(f"ΔC por sítio: mínimo {dC.min():,.0f} | máximo {dC.max():,.0f} $/h "
      f"(razão {dC.max()/dC.min():.1f}×)")
print(f"Anualizado: melhor sítio {dC.max()*8760/1e6:.1f} M$/ano | "
      f"pior {dC.min()*8760/1e6:.1f} M$/ano")
mapa_da_rede(net, dC, "A VERDADE física: ΔC de 50 MW de GD em cada barra [$ /h]",
             cmap="viridis", marcas=dC.nlargest(3).index.tolist())

### C.1 — O confronto: quem previu a verdade?

Calculem a correlação de Spearman entre $\Delta C$ (a verdade) e cada um dos quatro
candidatos a "pontuação barata": o índice $s$, a pontuação **social** (colunas de $\hat A$),
a **própria** (diagonal) e... o humilde **LMP médio** do painel.

In [ ]:
# ╔══ COMPLETAR ══ dica: st.spearmanr(dC, vetor).statistic para cada candidato;
# ║   montem um DataFrame com as quatro correlações, ordenado
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")


In [ ]:

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
sp = dict(zip(confronto["pontuação"], confronto["Spearman com ΔC"]))
assert sp["LMP médio"] > sp["índice s"] > 0.15, \
    "o LMP médio deveria vencer o índice, e ambos serem positivos"
assert abs(sp["social (col. de Â)"]) < 0.25, \
    "a pontuação social topológica NÃO deveria prever o ΔC físico"
print("✅ VERIFICAÇÃO OK")
display(confronto.round(2))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
axes[0].barh(confronto["pontuação"][::-1], confronto["Spearman com ΔC"][::-1],
             color=[CINZA, CINZA, AZUL, VERMELHO])
axes[0].axvline(0, color="k", lw=0.6)
axes[0].set_xlabel("Spearman com a verdade ΔC")
axes[0].set_title("Quem previu a física?")
axes[1].scatter(lmp_medio, dC, s=50, c=AZUL, edgecolors="k")
for b in dC.nlargest(5).index:
    axes[1].annotate(str(b), (lmp_medio[b], dC[b]),
                     textcoords="offset points", xytext=(5, 3), fontsize=8)
axes[1].set_xlabel("LMP médio do painel [$/MWh]")
axes[1].set_ylabel("ΔC verdadeiro [$/h]")
axes[1].set_title(f"O dual previu o valor do sítio (ρ_S = {sp['LMP médio']:+.2f})")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

💡 Comente os resultados anteriores



📝 **Pergunta C:** a barra 15 (campeã do ranking social) ficou em que posição no ranking da
verdade? Expliquem o resultado da aposta do Bloco B com os dois canais acima.

> *Resposta de vocês aqui:* ______

---
## Bloco D · Dois objetivos, uma recomendação

O índice $s$ e o $\Delta C$ **não medem a mesma coisa** — e está tudo bem: $s$ é
multi-critério (recurso renovável, demanda local, congestão, preço); $\Delta C$ é só custo
de operação. A recomendação profissional sai do **cruzamento**: a *shortlist* das barras
boas nos **dois** eixos.

In [ ]:
# ╔══ COMPLETAR ══ dica: shortlist = interseção (set &) entre os índices de
# ║   s.nlargest(10) e dC.nlargest(10)
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert len(shortlist) >= 2 and 2 in shortlist, \
    "a shortlist deveria ter ≥ 2 barras e incluir a barra 2"
print(f"✅ VERIFICAÇÃO OK — shortlist (top-10 nos DOIS critérios): {shortlist}")

fig, ax = plt.subplots(figsize=(8.5, 6))
ax.scatter(s, dC, s=55, c=CINZA, edgecolors="k", alpha=0.7)
ax.scatter(s[shortlist], dC[shortlist], s=160, c=VERDE, edgecolors="k", zorder=3,
           label="shortlist (boa nos dois)")
ax.axvline(s.nlargest(10).min(), color=AZUL, ls=":", lw=1, label="corte top-10 índice")
ax.axhline(dC.nlargest(10).min(), color=VERMELHO, ls=":", lw=1, label="corte top-10 ΔC")
for b in list(shortlist) + [int(s.idxmax()), int(dC.idxmax())]:
    ax.annotate(str(b), (s[b], dC[b]), textcoords="offset points",
                xytext=(6, 4), fontsize=9)
ax.set_xlabel("índice de aptidão s (multi-critério)")
ax.set_ylabel("ΔC verdadeiro [$/h] (custo de operação)")
ax.set_title("Dois objetivos, um cruzamento: a shortlist defensável")
ax.legend(); plt.tight_layout(); plt.show()

print(f"O que está em jogo: o melhor sítio vale {dC.max()*8760/1e6:.1f} M$/ano; "
      f"o pior, {dC.min()*8760/1e6:.1f} M$/ano — a escolha importa {dC.max()/dC.min():.0f}×.")

💡 Comente os resultados anteriores



📝 **Pergunta D (o entregável):** redijam a recomendação final (½ página): (i) a barra (ou
2 barras) recomendada(s) e as duas evidências; (ii) uma frase para o colega que escolheria
só pelo ranking social topológico; (iii) quando o índice multi-critério deve *vencer* o
$\Delta C$ na decisão (dica: o que o $\Delta C$ de operação não vê? recurso primário,
perdas, emissões, custo de conexão...).

> *Resposta de vocês aqui:* ______

